### Generate the INPUT file for each GW events to use SNANA to simulate kilonova light curves.

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re
import csv
import pprint

template_path = Path("<BASE_DIR>/ML+GW+KN/dataset/KN_sim/SIM_INPUT/SIMGEN_KN_LSST_TEMPLATE.INPUT")
simlib_dir = "<BASE_DIR>/data/simlib/"

In [ ]:
injections = pd.read_csv("<BASE_DIR>/ML+GW+KN/dataset/O5_sim_bns/injections_final.csv")
sim_id = 6


In [ ]:
# get NLIBID from simlib file
def get_NLIBID(simlib_file):
    nlibid = []
    with open(simlib_file, 'r') as f:
        lines = f.readlines()
        for line in lines:
            if line.startswith('NLIBID:'):
                nlibid = int(line.split()[1])
                break
    return nlibid

NLIBID = get_NLIBID(os.path.join(simlib_dir, f"baseline_v5.0.1_10yrs_{sim_id}.SIMLIB"))
print(f"NLIBID from simlib: {NLIBID}")

In [ ]:
template_text = template_path.read_text()
pprint.pprint(template_text)

In [ ]:
# replace NLIBID and GENVERSION in the template
text = template_text
NGENTOT_LC_line = f"NGENTOT_LC: {NLIBID}"
text = re.sub(
        r"^(NGENTOT_LC:\s*)\S+.*$",
        rf"\1 {NLIBID}",
        text,
        flags=re.MULTILINE
)
genversion_new = f"MY_LSST_KN_{sim_id}"
text = re.sub(
    r"^(GENVERSION:\s*)\S+.*$",
    rf"\1{genversion_new}",
    text,
    flags=re.MULTILINE
)

In [ ]:
idx = np.where(injections['simulation_id']==sim_id)[0]
mjd_explode = injections['mjd_time'].iloc[idx].values
costheta = injections['costheta'].iloc[idx].values
phi = injections['phi'].iloc[idx].values
mej_dyn = injections['mej_dyn'].iloc[idx].values
mej_wind = injections['mej_wind'].iloc[idx].values

# modify explosion time and ejecta parameters
text = re.sub(
    r"^(MJD_EXPLODE:\s*)\S+.*$",
    rf"\1 {mjd_explode[0]}",
    text,
    flags=re.MULTILINE
)
text = re.sub(
    r"^(GENPEAK_COSTHETA:\s*)\S+.*$",
    rf"\1 {costheta[0]}",
    text,
    flags=re.MULTILINE
)
text = re.sub(
    r"^(GENPEAK_PHI:\s*)\S+.*$",
    rf"\1 {phi[0]}",
    text,
    flags=re.MULTILINE
)
text = re.sub(
    r"^(GENPEAK_MEJDYN:\s*)\S+.*$",
    rf"\1 {mej_dyn[0]}",
    text,
    flags=re.MULTILINE
)
text = re.sub(
    r"^(GENPEAK_MEJWIND:\s*)\S+.*$",
    rf"\1 {mej_wind[0]}",
    text,
    flags=re.MULTILINE
)
# MODIFY SIMLIB FILE
text = re.sub(
    r"^(SIMLIB_FILE:\s*.+)_(?=\.SIMLIB)",
    fr"\1_{sim_id}",
    text,
    flags=re.MULTILINE
)

pprint.pprint(text)

In [ ]:
outfile = f"<BASE_DIR>/data/SIM_INPUT/SIMGEN_KN_LSST_{sim_id}.INPUT"
with open(outfile, "w", encoding="utf-8") as f:
    f.write(text)